> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 冲突检测与解决（Conflict Detection and Resolution）

## 概览

现代数据流水线——尤其是构建知识图谱的——经常从多个异构来源摄取数据（内部数据库、第三方 API、网页抓取）。分歧在所难免。

**Semantica 冲突解决模块**（`semantica.conflicts`）为管理这些数据不一致提供了坚实的框架，确保下游应用只消费高质量、已调和的数据。

> 译注：这正是 Semantica 的一条工程纪律——**冲突事实要先经 conflicts 标记、deduplication 去重，才能进图谱；禁止静默覆盖**。本篇是这条纪律的前半步（标记），18 篇去重是后半步（合并）。

### 什么算"冲突"？

- **同一实体的多条记录**在某个字段上不一致，就是冲突。
- Semantica 假定每条记录是一个字典：
  - `id`（或 `entity_id`）：所描述实体的稳定标识
  - 一个或多个属性（如 `name`、`birth_date`、`department`）
  - `source`：值从哪来（db、scrape、api、file）
  - 可选 `timestamp`：值被观察到的时间
- 模块感知来源：能记录**哪个来源贡献了哪个值**，再用投票或可信度等策略解决。

### 实用 API 提示（避开常见错配）

- 想检查单个字段，用 `ConflictDetector.detect_value_conflicts(entities, property_name=...)`。
- 想做更宽的扫描（值/类型/时间等），用 `ConflictDetector.detect_conflicts(entities)`。
- 如果手里是 `GraphBuilder` 建的知识图谱字典，用 `kg.get("entities", [])` 取实体。

### 先看主线

02 篇的九种摄取源进来，同一实体难免说法不一。本篇用"同一员工的三份档案"贯穿六步，像一场完整的审理——**布局 → 找茬 → 拍板**：

**布局（第 1–2 步）—— 开局**

1. 第 1 步 备料：三份带冲突的员工档案（冲突是刻意埋好的）
2. 第 2 步 定信源：给三个来源注册可信度——先立"听谁的"规矩

**找茬（第 3–4 步）—— 审理**

3. 第 3 步 检测：`ConflictDetector` 找出打架的字段
4. 第 4 步 分析：`ConflictAnalyzer` 看规律——是不是某个来源惯犯

**拍板（第 5–6 步）—— 判决**

5. 第 5 步 解决：两种判决法——投票（人人平等）vs 可信度加权（信权威）
6. 第 6 步 交人：判不了的生成调查指南，人工复核兜底

### 核心能力

1. **多维冲突检测**
    *   **值冲突**：同一属性值不同（如 `"Google"` vs `"Google Inc."`）
    *   **类型冲突**：数据类型不匹配（如字符串 vs 整数）
    *   **时间冲突**：时间线不一致（如 `start_date` 晚于 `end_date`）

2. **溯源与来源追踪**
    *   **细粒度追踪**：每个属性值都能追回具体的来源文档、页面或 API 调用
    *   **可信度打分**：给来源分配信任分（如内部 HR 库 `0.95`、网页抓取 `0.60`）

3. **自动解决策略**
    *   **投票**：多数说了算（适合多个等权来源）
    *   **可信度加权**：高信任来源的值压过其他
    *   **新近优先**：最新的数据点获胜
    *   **专家复核**：复杂冲突标记给人处理

4. **调查与审计**
    *   **调查指南**：自动生成面向分析人员的分步指南
    *   **审计留痕**：记录每个冲突如何被解决，满足合规

## 安装

确认环境里装好 Semantica：

```bash
pip install semantica
```

- 在 Semantica 仓库里跑这个 notebook，建议可编辑安装（代码改动立即生效）：
  - `pip install -e .`
- 在托管 notebook 环境里，`%pip install semantica` 通常比 `!pip install ...` 可靠——它装进当前内核。

In [ ]:
!pip install -q semantica 

In [ ]:
import json
from datetime import datetime

## 第 1 步：模拟多源数据

审理开局，先造案情。用员工数据模拟一个真实场景。

**场景：**

**员工 001** 的记录来自三个来源：

- **HR 数据库**：高信任的内部来源
- **LinkedIn 抓取**：可靠性较低的外部来源
- **公共名录**：过时的公共 API

**记录的形状（Semantica 期望的样子）：**

- 每条记录描述同一实体（`id`: `emp_001`）的字典
- 每条带 `source` 键，冲突才能归因
- `timestamp` 让你能用基于时间的解决策略（如最新获胜）

**埋好的冲突：**
*   **`birth_date`**：公共名录的年份不一样
*   **`department`**：LinkedIn 用了更细的名称（"Software Engineering"），HR 库是泛称 "Engineering"

In [ ]:
# 1. Define source metadata
sources_metadata = {
    "hr_db": {"credibility": 0.95, "type": "internal_database"},
    "linkedin_scrape": {"credibility": 0.60, "type": "web_scrape"},
    "public_dir": {"credibility": 0.40, "type": "public_api"}
}

# 2. Define entity records from these sources
entity_records = [
    {
        "id": "emp_001",
        "name": "John Doe",
        "birth_date": "1980-05-15",
        "department": "Engineering",
        "source": "hr_db",
        "timestamp": "2023-01-01T10:00:00"
    },
    {
        "id": "emp_001",
        "name": "Jonathan Doe",
        "birth_date": "1980-05-15",
        "department": "Software Engineering",
        "source": "linkedin_scrape",
        "timestamp": "2023-06-15T14:30:00"
    },
    {
        "id": "emp_001",
        "name": "John Doe",
        "birth_date": "1982-05-15",  # Conflict: Different year
        "department": "Engineering",
        "source": "public_dir",
        "timestamp": "2022-12-01T09:00:00"
    }
]

print(f"Loaded {len(entity_records)} records for Employee 001")

## 第 2 步：注册与追踪来源

要按信任度解决冲突，得先把来源注册进 `SourceTracker`。

`SourceTracker` 是中央登记处，管：

- **可信度分数**：你有多信这个来源
- **元数据**：有用的上下文（来源类型、权威系统还是抓取）

**可信度分数怎么想：**

- 把分数当*相对排序*用——小数点后的精确值不如排名重要
- 从简单开始：`内部库 > 厂商 API > 网页抓取`
- 之后用分析数据回调（"哪个来源经常错？"）

下面把模拟的来源逐个注册。

In [ ]:
from semantica.conflicts import SourceTracker

source_tracker = SourceTracker()

print("Registering sources...")
for source_id, metadata in sources_metadata.items():
    source_tracker.register_source(
        source_id=source_id,
        source_type=metadata["type"],
        credibility_score=metadata["credibility"]
    )
    print(f"  - Registered '{source_id}' with credibility {metadata['credibility']}")

## 第 3 步：检测冲突

用 `ConflictDetector` 扫描记录找分歧。

检测器很灵活，可以配置成：

- **检查特定属性**：只查 `birth_date` 这类关键字段
- **实体级全扫**：扫一个实体的多个（或全部）属性

**返回什么：**

- 一列 `Conflict` 对象
- 常看字段：
  - `conflict_type`（如 `value_conflict`）
  - `entity_id`、`property_name`
  - `conflicting_values` 和 `sources`
  - `severity` 和 `confidence`

这里明确检查 `birth_date` 和 `department`。

In [ ]:
from semantica.conflicts import ConflictDetector

# Initialize detector with our populated source tracker
detector = ConflictDetector(source_tracker=source_tracker)

conflicts = []

# 1. Check birth_date
dob_conflicts = detector.detect_value_conflicts(entity_records, "birth_date")
conflicts.extend(dob_conflicts)

# 2. Check department
dept_conflicts = detector.detect_value_conflicts(entity_records, "department")
conflicts.extend(dept_conflicts)

print(f"Detected {len(conflicts)} conflicts:")
for conflict in conflicts:
    print(f"- {conflict.conflict_type.value}: {conflict.property_name} for {conflict.entity_id}")
    print(f"  Values: {conflict.conflicting_values}")
    print(f"  Severity: {conflict.severity}")
    print("--- ")

## 第 4 步：分析冲突模式

数据量一大，单条冲突不如系统性规律重要。`ConflictAnalyzer` 回答这类问题：

- "是不是某个来源贡献了大部分冲突？"
- "冲突集中在某类实体或某个属性上吗？"
- "冲突严重度和类型的分布如何？"

**怎么用进流水线：**

- 跑分析，找出噪声来源
- 用结果回调第 2 步的可信度分数，或改进摄取/清洗规则
- 跟踪趋势，及时发现上游系统的退化

In [ ]:
from semantica.conflicts import ConflictAnalyzer

analyzer = ConflictAnalyzer()
analysis = analyzer.analyze_conflicts(conflicts)

print("Conflict Analysis Summary:")
print(f"Total Conflicts: {analysis['total_conflicts']}")
print(f"By Type: {analysis.get('by_type', {}).get('counts')}")
print(f"By Severity: {analysis.get('by_severity', {}).get('counts')}")

## 第 5 步：解决冲突

关键一步：决定信哪个值。Semantica 提供灵活的解决策略。

### 策略 A：投票（多数说了算）

选出现最频繁的值。简单，但所有来源一律平等。

- 适合：多个质量相近的独立来源
- 不适合：有单一权威系统应当永远胜出时

### 策略 B：可信度加权

按来源的 `credibility` 给每个值算加权分。

**例：**
*   `hr_db`（0.95）说 "1980-05-15"
*   `public_dir`（0.40）说 "1982-05-15"

哪怕多个低质来源在错误日期上一致，高可信来源大概率获胜。

**解决器返回什么：**

- 一列解决结果，每项通常包含：
  - 是否已解决
  - 选中的值（`resolved_value`）
  - 置信度分数
  - 元数据（如属性名），支撑审计留痕

运行下一个 cell，对比投票与可信度加权的结局。

In [ ]:
from semantica.conflicts import ConflictResolver

resolver = ConflictResolver()

# CRITICAL: Link the source tracker to the resolver.
# This allows the resolver to look up the credibility scores we registered in Step 2.
resolver.set_source_tracker(source_tracker)

print("--- Resolution: Voting ---")
voting_results = resolver.resolve_conflicts(conflicts, strategy="voting")
for res in voting_results:
    print(f"Property: {res.metadata.get('property_name'):<15} | Resolved Value: {res.resolved_value}")

print("\n--- Resolution: Credibility Weighted ---")
# Notice how the HR DB's value is preferred due to higher credibility
credibility_results = resolver.resolve_conflicts(conflicts, strategy="credibility_weighted")
for res in credibility_results:
    print(f"Property: {res.metadata.get('property_name'):<15} | Resolved Value: {res.resolved_value} (Confidence: {res.confidence:.2f})")

## 第 6 步：生成调查指南

不是所有冲突都能自动解决。高风险或低置信度的裁决需要人工复核。

`InvestigationGuideGenerator` 给分析师生成结构化的"飞行计划"，写明：

- **什么**在冲突（实体 + 字段 + 竞争值）
- **谁**卷入（哪个来源产出了哪个值）
- **怎么**核实正确数据（分析人员可执行的步骤）

**什么时候生成指南：**

- 低置信度的解决结果
- 关键字段上的冲突（身份、法定名称、合规属性）
- 任何想在写回图谱前设一道人工关卡的场合

In [ ]:
from semantica.conflicts import InvestigationGuideGenerator

guide_generator = InvestigationGuideGenerator()

# Generate a guide for the first conflict (birth_date)
guide = guide_generator.generate_guide(conflicts[0])

print(f"=== {guide.title} ===")
print(f"Summary: {guide.conflict_summary}\n")

print("Investigation Steps:")
for i, step in enumerate(guide.investigation_steps, 1):
    print(f"{i}. {step.description}")
    print(f"   Action: {step.action}")

print("\nRecommended Actions:")
for action in guide.recommended_actions:
    print(f"[ ] {action}")

## 结语

你用 Semantica 建成了一条冲突解决流水线！

**回顾主线**

一场完整审理走完：**布局**（备料三份档案、给来源立可信度规矩）→ **找茬**（检测出打架的字段、分析出规律）→ **拍板**（投票与可信度加权两种判决，判不了的生成调查指南交人）。没有静默覆盖——每个决定留痕、可审计。下一步把已标记的冲突交给 18 篇去重合并，数据才能干净进图谱。

**成果清单：**

- **模拟**了带真实分歧的多源实体记录
- **注册**了带可信度分数的来源，建立信任层级
- **检测**了特定属性的值冲突
- **分析**了冲突的类型与严重度分布
- **解决**了冲突（投票与可信度加权）
- **生成**了支撑人工复核的调查指南

**真实项目的建议下一步：**

- 接入摄取层，让每个抽出的值都带 `source` 和（最好）`timestamp`
- 用 `ConflictDetector.detect_conflicts(...)` 扩展到值冲突之外
- 保存解决结果和指南产出，为下游建审计链